In [0]:
# Pipeline Orchestrator
# This notebook runs the complete TechMart Catalog Intelligence Pipeline
# from Bronze ingestion to Gold aggregation in a single execution.
# Run this notebook to execute the full end-to-end pipeline.

print("""
╔══════════════════════════════════════════════════════╗
║   TechMart Catalog Intelligence Pipeline             ║
║   End-to-End Orchestrator                            ║
╚══════════════════════════════════════════════════════╝
""")

In [0]:
# Bronze: Raw ingestion
# Reads raw Excel files from Unity Catalog Volume
# Writes raw_products and raw_vendors Delta tables

print("=" * 55)
print("STAGE 1 — Bronze Ingestion")
print("=" * 55)

dbutils.notebook.run("01_bronze_ingest", timeout_seconds=300)

print("✅ Bronze ingestion complete")

In [0]:
# Silver: Standardization
# Normalizes weight, price, vendor names
# Writes silver products and vendors Delta tables

print("=" * 55)
print("STAGE 2 — Silver Standardization")
print("=" * 55)

dbutils.notebook.run("02_silver_standardize", timeout_seconds=300)

print("✅ Silver standardization complete")

In [0]:
%pip install pydantic

In [0]:
# Silver: LLM Extraction
# Calls Groq API to extract name, brand, sub-category
# Logs results to MLflow for traceability

print("=" * 55)
print("STAGE 3 — LLM Extraction")
print("=" * 55)

dbutils.notebook.run("03_llm_extraction", timeout_seconds=600)

print("✅ LLM extraction complete")

In [0]:
# CELL — Stage 3 with detailed error capture
import traceback

try:
    result = dbutils.notebook.run(
        f"/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/pipeline/03_llm_extraction", 
        timeout_seconds=600
    )
    print("✅ LLM extraction complete")
except Exception as e:
    print(f"❌ Stage 3 failed")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    # This forces the pipeline to stop and show the real error
    raise

In [0]:
# Silver: LLM Judge
# Validates extraction against approved taxonomy
# Flags records that don't meet quality standards

print("=" * 55)
print("STAGE 4 — LLM Judge")
print("=" * 55)

dbutils.notebook.run("./04_llm_judge", timeout_seconds=600)

print("✅ LLM judge complete")

In [0]:
# Silver: Taxonomy Enrichment
# Joins LLM results with vendor info and category mapping

print("=" * 55)
print("STAGE 5 — Silver Taxonomy Enrichment")
print("=" * 55)

dbutils.notebook.run("05_silver_taxonomy", timeout_seconds=300)

print("✅ Silver taxonomy enrichment complete")

In [0]:
# Gold: Aggregation
# Produces final business-ready aggregated table

print("=" * 55)
print("STAGE 6 — Gold Aggregation")
print("=" * 55)

dbutils.notebook.run("06_gold_aggregation", timeout_seconds=300)

print("✅ Gold aggregation complete")

In [0]:
# Pipeline Summary
# Shows final row counts for all tables produced

print("""
╔══════════════════════════════════════════════════════╗
║   Pipeline Complete — Final Summary                  ║
╚══════════════════════════════════════════════════════╝
""")

tables = {
    "Bronze — raw_products"       : "main.bronze_techmart.raw_products",
    "Bronze — raw_vendors"        : "main.bronze_techmart.raw_vendors",
    "Silver — products"           : "main.silver_techmart.products",
    "Silver — vendors"            : "main.silver_techmart.vendors",
    "Silver — llm_extracted"      : "main.silver_techmart.llm_extracted",
    "Silver — taxonomy"           : "main.silver_techmart.taxonomy",
    "Silver — taxonomy_enriched"  : "main.silver_techmart.taxonomy_enriched",
    "Gold   — product_summary"    : "main.gold_techmart.product_summary",
}

for label, table in tables.items():
    try:
        count = spark.table(table).count()
        print(f"  {label:<35} {count} rows")
    except:
        print(f"  {label:<35} ⚠️ table not found")